In [ ]:
# code to convert the masks into the yolo format

import os
import cv2

# Set up paths
image_dir = r"C:\Medical_image_analysis\kvasir_seg_sessile\images"
mask_dir = r"C:\Medical_image_analysis\kvasir_seg_sessile\masks"
label_dir = r"C:\Medical_image_analysis\kvasir_seg_sessile\labels"

# Create the labels folder if it doesn't exist
os.makedirs(label_dir, exist_ok=True)

# Get all image files
image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

# Loop through all images
for file in image_files:
    img_path = os.path.join(image_dir, file)
    mask_path = os.path.join(mask_dir, file)
    label_path = os.path.join(label_dir, os.path.splitext(file)[0] + ".txt")

    # Load image and mask
    image = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    height, width = image.shape[:2]

    # Threshold the mask in case it's not binary
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # Find all external contours (each polyp)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    with open(label_path, 'w') as label_file:
        for cnt in contours:
            x, y, w_box, h_box = cv2.boundingRect(cnt)

            # Optional: ignore small bounding boxes (noise)
            if w_box < 5 or h_box < 5:
                continue

            # Convert to YOLO format (normalized)
            x_center = (x + w_box / 2) / width
            y_center = (y + h_box / 2) / height
            norm_w = w_box / width
            norm_h = h_box / height

            # Class ID is 0 (for polyp)
            yolo_line = f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
            label_file.write(yolo_line + "\n")


In [1]:
#training with dataset bkai with clahe only no augmentation no negative

from ultralytics import YOLO

# Load YOLOv8-Large model (pretrained)
model = YOLO("yolov8m.pt")  # use 'yolov8l.yaml' if training from scratch

# Train the model
model.train(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    name='yolov8_m_augmentation_CIoU',  # updated to reflect large model and today's date
    project=r"C:\Medical_image_analysis\yolov8_kvasir\results",
    workers=4,
    verbose=True,
    patience=50
)

New https://pypi.org/project/ultralytics/8.3.167 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_m_augmentation

train: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\train.cache... 7700 images, 700 backgrounds, 0 corrupt: 100%|██████████| 7700/7700 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.30.1 ms, read: 117.622.3 MB/s, size: 37.3 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\val.cache... 200 images, 20 backgrounds, 0 corrupt: 100%|██████████| 220/220 [00:00<?, ?it/s]


Plotting labels to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        220        214      0.955      0.883      0.947      0.801
EarlyStopping: Training stopped early as no improvement observed in last 50 epochs. Best results observed at epoch 88, best model saved as best.pt.
To update EarlyStopping(patience=50) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



138 epochs completed in 4.874 hours.
Optimizer stripped from C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU\weights\last.pt, 52.0MB
Optimizer stripped from C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU\weights\best.pt, 52.0MB

Validating C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU\weights\best.pt...
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:03<00:00,  2.28it/s]


                   all        220        214      0.947      0.907      0.965       0.82
Speed: 0.2ms preprocess, 3.8ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_m_augmentation_CIoU


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D38EF8F250>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

## CODE THAT RUN ON base-env Kernel that contains CIoU loss

In [1]:
#training with dataset bkai with clahe only no augmentation no negative

from ultralytics import YOLO

# Load YOLOv8-Large model (pretrained)
model = YOLO("yolov8s.pt")  # use 'yolov8l.yaml' if training from scratch

# Train the model
model.train(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    name='yolov8_s_augmentation_CIoU',  # updated to reflect large model and today's date
    project=r"C:\Medical_image_analysis\yolov8_kvasir\results",
    workers=4,
    verbose=True,
    patience=50
)

New https://pypi.org/project/ultralytics/8.3.167 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_s_augmentation

train: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\train.cache... 7700 images, 700 backgrounds, 0 corrupt: 100%|██████████| 7700/7700 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.40.0 ms, read: 52.17.2 MB/s, size: 37.3 KB)


val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\val.cache... 200 images, 20 backgrounds, 0 corrupt: 100%|██████████| 220/220 [00:00<?, ?it/s]


Plotting labels to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        220        214      0.941      0.897      0.951      0.791
EarlyStopping: Training stopped early as no improvement observed in last 50 epochs. Best results observed at epoch 77, best model saved as best.pt.
To update EarlyStopping(patience=50) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



127 epochs completed in 2.920 hours.
Optimizer stripped from C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU\weights\last.pt, 22.5MB
Optimizer stripped from C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU\weights\best.pt, 22.5MB

Validating C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU\weights\best.pt...
Ultralytics 8.3.152  Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:02<00:00,  2.79it/s]


                   all        220        214      0.966      0.874      0.963      0.809
Speed: 0.2ms preprocess, 2.2ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to C:\Medical_image_analysis\yolov8_kvasir\results\yolov8_s_augmentation_CIoU


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001EB8893DED0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

## CIoU + No augmentations + No negatives

In [20]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_l_kvasir_no_aug_no_negative_CIoU\yolov12_l_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 264.035.5 MB/s, size: 31.2 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 100.3Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.4it/s 5.2s0.4s7
                   all        100        114      0.937      0.807      0.858      0.703
Speed: 1.8ms preprocess, 15.4ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val61
mAP@0.5: 0.8585
mAP@0.5:0.95: 0.7032
Precision: 0.9371
Recall: 0.8070
F1-score: 0.8672


In [21]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_m_kvasir_no_aug_no_negative_CIoU\yolov12_m_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12m summary (fused): 169 layers, 20,105,683 parameters, 0 gradients, 67.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 288.5100.0 MB/s, size: 33.1 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.5s0.3s5
                   all        100        114      0.858      0.795      0.828      0.695
Speed: 1.6ms preprocess, 10.0ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val62
mAP@0.5: 0.8276
mAP@0.5:0.95: 0.6947
Precision: 0.8580
Recall: 0.7948
F1-score: 0.8252


In [22]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_s_kvasir_no_aug_no_negative_CIoU\yolov12_s_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12s summary (fused): 159 layers, 9,231,267 parameters, 0 gradients, 21.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 273.054.0 MB/s, size: 36.0 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.5s0.2s8
                   all        100        114      0.893      0.833      0.858      0.728
Speed: 1.6ms preprocess, 7.3ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val63
mAP@0.5: 0.8581
mAP@0.5:0.95: 0.7281
Precision: 0.8929
Recall: 0.8333
F1-score: 0.8621


In [23]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 565.7372.1 MB/s, size: 78.6 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.3s0.2s5
                   all        100        114      0.905      0.831      0.868      0.719
Speed: 1.9ms preprocess, 4.5ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val64
mAP@0.5: 0.8682
mAP@0.5:0.95: 0.7187
Precision: 0.9045
Recall: 0.8312
F1-score: 0.8663


In [24]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 293.352.9 MB/s, size: 33.1 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 100.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.5s0.2s8
                   all        100        114      0.905      0.831      0.868      0.719
Speed: 2.0ms preprocess, 5.0ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val65
mAP@0.5: 0.8682
mAP@0.5:0.95: 0.7187
Precision: 0.9045
Recall: 0.8312
F1-score: 0.8663


## YOLOv12 Kvasir-seg (CIoU + No Augmentations + No Negatives) for 100 test images

In [14]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_l_kvasir_no_aug_no_negative_CIoU\yolov12_l_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 252.6120.0 MB/s, size: 29.9 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.4it/s 4.9s0.4s2
                   all        100        114      0.937      0.807      0.858      0.703
Speed: 1.7ms preprocess, 15.5ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val56
mAP@0.5: 0.8585
mAP@0.5:0.95: 0.7032
Precision: 0.9371
Recall: 0.8070
F1-score: 0.8672


In [15]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_m_kvasir_no_aug_no_negative_CIoU\yolov12_m_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12m summary (fused): 169 layers, 20,105,683 parameters, 0 gradients, 67.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 281.2112.4 MB/s, size: 35.1 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 192.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.5s0.3s4
                   all        100        114      0.858      0.795      0.828      0.695
Speed: 1.7ms preprocess, 10.2ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val57
mAP@0.5: 0.8276
mAP@0.5:0.95: 0.6947
Precision: 0.8580
Recall: 0.7948
F1-score: 0.8252


In [16]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_s_kvasir_no_aug_no_negative_CIoU\yolov12_s_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12s summary (fused): 159 layers, 9,231,267 parameters, 0 gradients, 21.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 281.248.4 MB/s, size: 31.4 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 99.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.5s0.2s6
                   all        100        114      0.893      0.833      0.858      0.728
Speed: 1.6ms preprocess, 6.0ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val58
mAP@0.5: 0.8581
mAP@0.5:0.95: 0.7281
Precision: 0.8929
Recall: 0.8333
F1-score: 0.8621


In [17]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv12\YOLOv12_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\yolov12_n_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 195.365.6 MB/s, size: 24.5 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 99.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.5s0.2s8
                   all        100        114      0.905      0.831      0.868      0.719
Speed: 1.6ms preprocess, 6.1ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val59
mAP@0.5: 0.8682
mAP@0.5:0.95: 0.7187
Precision: 0.9045
Recall: 0.8312
F1-score: 0.8663


## YOLOv8 Kvasir-seg (CIoU + No Augmentations + No Negatives) for 100 test images

In [10]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\YOLOv8_kvasir-seg_no_aug_no_negative_CIoU\yolov8_l_kvasir_no_aug_no_negative_CIoU\yolov8_l_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 112 layers, 43,607,379 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 219.658.2 MB/s, size: 31.2 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.4it/s 4.9s0.4s3
                   all        100        114      0.894      0.833      0.845      0.692
Speed: 1.6ms preprocess, 13.5ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val52
mAP@0.5: 0.8448
mAP@0.5:0.95: 0.6925
Precision: 0.8938
Recall: 0.8333
F1-score: 0.8625


In [11]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\YOLOv8_kvasir-seg_no_aug_no_negative_CIoU\yolov8_m_kvasir_no_aug_no_negative_CIoU\yolov8_m_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 263.996.3 MB/s, size: 32.0 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 28.3Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.5it/s 4.6s0.3s8
                   all        100        114      0.902      0.842       0.86      0.711
Speed: 1.6ms preprocess, 8.7ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val53
mAP@0.5: 0.8601
mAP@0.5:0.95: 0.7114
Precision: 0.9017
Recall: 0.8421
F1-score: 0.8709


In [12]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\YOLOv8_kvasir-seg_no_aug_no_negative_CIoU\yolov8_s_kvasir_no_aug_no_negative_CIoU\yolov8_s_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 305.654.2 MB/s, size: 34.6 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 86.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.7it/s 4.2s0.2ss
                   all        100        114      0.905      0.835      0.876      0.723
Speed: 1.7ms preprocess, 3.9ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val54
mAP@0.5: 0.8759
mAP@0.5:0.95: 0.7230
Precision: 0.9050
Recall: 0.8353
F1-score: 0.8688


In [13]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\yolov8_kvasir\results\YOLOv8\YOLOv8_kvasir-seg_no_aug_no_negative_CIoU\yolov8_n_kvasir_no_aug_no_negative_CIoU\yolov8_n_kvasir_no_aug_no_negative_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\yolov8_kvasir\kvasir.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 274.464.6 MB/s, size: 32.5 KB)
val: Scanning C:\Medical_image_analysis\yolov8_kvasir\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 91.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.5it/s 4.5s0.2s8
                   all        100        114      0.942      0.825      0.874       0.73
Speed: 1.7ms preprocess, 4.8ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val55
mAP@0.5: 0.8738
mAP@0.5:0.95: 0.7301
Precision: 0.9418
Recall: 0.8246
F1-score: 0.8793


## YOLOv8 (CIoU + No Augmentations + No Negatives) for 100 test images ................(BKAI-IGH NeoPolyp Dataset)...................

In [5]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\yolov8_with_baseline_dataset_CIoU\yolov8_l_baseline_model_CIoU\yolov8_l_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 112 layers, 43,608,150 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1272.3183.8 MB/s, size: 265.2 KB)
val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 100.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.5it/s 4.7s0.3s9
                   all        100        115      0.793      0.805      0.813      0.694
            neoplastic         68         70       0.86        0.9       0.93      0.832
        non-neoplastic         39         45      0.727      0.709      0.696      0.556
Speed: 1.6ms preprocess, 11.0ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val47
mAP@0.5: 0.8131

In [6]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\yolov8_with_baseline_dataset_CIoU\yolov8_m_baseline_model_CIoU\yolov8_m_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1472.8179.2 MB/s, size: 275.7 KB)
val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 100.1Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.5it/s 4.6s0.3s6
                   all        100        115       0.82      0.761      0.811      0.711
            neoplastic         68         70      0.897        0.9      0.931      0.852
        non-neoplastic         39         45      0.742      0.622      0.691      0.569
Speed: 1.6ms preprocess, 7.6ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val48
mAP@0.5: 0.8112
mA

In [7]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\yolov8_with_baseline_dataset_CIoU\yolov8_s_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1416.4254.1 MB/s, size: 276.9 KB)
val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 149.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.5it/s 4.6s0.2s0
                   all        100        115      0.801      0.813      0.843      0.728
            neoplastic         68         70      0.912      0.893      0.947       0.85
        non-neoplastic         39         45       0.69      0.733      0.739      0.606
Speed: 1.5ms preprocess, 6.2ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val49
mAP@0.5: 0.8431
mA

In [8]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\yolov8_with_baseline_dataset_CIoU\yolov8_n_baseline_model_CIoU\yolov8_n_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1439.3141.8 MB/s, size: 328.3 KB)
val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 99.3Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.4s0.2s8
                   all        100        115      0.766      0.803      0.823      0.696
            neoplastic         68         70      0.789      0.963      0.924      0.825
        non-neoplastic         39         45      0.743      0.642      0.722      0.567
Speed: 1.6ms preprocess, 3.1ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val50
mAP@0.5: 0.8231
mAP@0

## YOLOv12 (CIoU + No Augmentations + No Negatives) for 100 test images ................(BKAI-IGH NeoPolyp Dataset)...................

In [4]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\YOLO12\YOLOV12_baseline_model_CIoU\yolov12_l_baseline_model_CIoU\yolov12_l_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12l summary (fused): 283 layers, 26,340,614 parameters, 0 gradients, 88.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1347.675.6 MB/s, size: 263.7 KB)
val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 23.3Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.4it/s 5.0s0.3s9
                   all        100        115      0.715      0.824      0.786       0.68
            neoplastic         68         70      0.737      0.943      0.906      0.833
        non-neoplastic         39         45      0.694      0.705      0.667      0.527
Speed: 1.4ms preprocess, 13.1ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val46
mAP@0.5: 0.7863

In [3]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\YOLO12\YOLOV12_baseline_model_CIoU\yolov12_m_baseline_model_CIoU\yolov12_m_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12m summary (fused): 169 layers, 20,106,454 parameters, 0 gradients, 67.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1214.0307.4 MB/s, size: 255.1 KB)
val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.5it/s 4.5s0.3s4
                   all        100        115      0.748      0.859       0.83      0.687
            neoplastic         68         70      0.816      0.914      0.926      0.807
        non-neoplastic         39         45       0.68      0.804      0.735      0.567
Speed: 1.5ms preprocess, 10.1ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val45
mAP@0.5: 0.8302
mAP@0.5

In [2]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\YOLO12\YOLOV12_baseline_model_CIoU\yolov12_s_baseline_model_CIoU\yolov12_s_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12s summary (fused): 159 layers, 9,231,654 parameters, 0 gradients, 21.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1172.1185.4 MB/s, size: 273.6 KB)
val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 188.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.4s0.2s2
                   all        100        115      0.761      0.803      0.839      0.738
            neoplastic         68         70      0.836      0.929      0.937      0.858
        non-neoplastic         39         45      0.685      0.678       0.74      0.619
Speed: 1.4ms preprocess, 6.9ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val44
mAP@0.5: 0.8385

In [1]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\results\YOLO12\YOLOV12_baseline_model_CIoU\yolov12_n_baseline_model_CIoU\yolov12_n_baseline_model_CIoU\weights\best.pt")
# Run validation on preprocessed Kvasir images
metrics = model.val(
    data=r"C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\data.yaml",
    split='test',
    device=0  # Use GPU; change to 'cpu' if needed
)

# Print evaluation metrics
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")
print(f"F1-score: {metrics.box.f1[0]:.4f}")

Ultralytics 8.3.228  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A5000, 23028MiB)
YOLOv12n summary (fused): 159 layers, 2,557,118 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1065.8128.0 MB/s, size: 260.7 KB)
val: Scanning C:\Medical_image_analysis\BKAI-IGH NeoPolyp\bkai-igh-neopolyp\yolov8_bkai\labels\test.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 4.3it/s 1.6s0.2s
                   all        100        115      0.808      0.752      0.799      0.677
            neoplastic         68         70      0.903      0.843      0.922      0.834
        non-neoplastic         39         45      0.713      0.662      0.675       0.52
Speed: 1.7ms preprocess, 6.9ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to C:\Users\Admin\Downloads\runs\detect\val43
mAP@0.5: 0.7986
mAP@0.5:0.9

In [13]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class PolypDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_filenames = sorted(os.listdir(images_dir))
        self.mask_filenames = sorted(os.listdir(masks_dir))
        self.transform = transform

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_filenames[idx])
        mask_path = os.path.join(self.masks_dir, self.mask_filenames[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Assuming masks are grayscale

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        return image, mask

# Example transforms - resize to 512x512, convert to tensor, normalize images
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # Using ImageNet mean/std for RGB normalization
                         std=[0.229, 0.224, 0.225]),
])

# Usage example for Kvasir-Seg dataset
kvasir_images_dir = r"C:\Medical_image_analysis\Polyp_domain_adaptation\kvasir-seg\images"
kvasir_masks_dir = r"C:\Medical_image_analysis\Polyp_domain_adaptation\kvasir-seg\images"

kvasir_dataset = PolypDataset(kvasir_images_dir, kvasir_masks_dir, transform=transform)


In [15]:
import os
import json
import numpy as np

# Paths
kvasir_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\raw\kvasir_on_combined"
bkai_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\raw\bkai_on_combined"
output_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\fused"

os.makedirs(output_path, exist_ok=True)

# Set weights for weighted average
weight_kvasir = 0.6   # change as per your choice
weight_bkai = 0.4     # sum of weights can be 1 or any numbers

# Get all JSON files (assuming same filenames in both folders)
json_files = sorted(os.listdir(kvasir_path))

for fname in json_files:
    kvasir_file = os.path.join(kvasir_path, fname)
    bkai_file = os.path.join(bkai_path, fname)

    # Load JSON predictions
    with open(kvasir_file, 'r') as f:
        kvasir_pred = json.load(f)

    with open(bkai_file, 'r') as f:
        bkai_pred = json.load(f)

    fused_pred = kvasir_pred.copy()
    fused_pred['preds'] = []

    # Assuming number of predictions (boxes) are the same and in same order
    for kv_box, bk_box in zip(kvasir_pred['preds'], bkai_pred['preds']):
        probs_kv = np.array(kv_box['probs'])
        probs_bk = np.array(bk_box['probs'])

        # Weighted average
        fused_probs = (weight_kvasir * probs_kv + weight_bkai * probs_bk) / (weight_kvasir + weight_bkai)

        # Copy bbox and class from one model (you can also implement bbox fusion later)
        fused_box = {
            'xyxy': kv_box['xyxy'],
            'conf': float(np.max(fused_probs)),  # new confidence = max prob
            'cls': int(np.argmax(fused_probs)),  # predicted class
            'probs': fused_probs.tolist()
        }

        fused_pred['preds'].append(fused_box)

    # Save fused JSON
    out_file = os.path.join(output_path, fname)
    with open(out_file, 'w') as f:
        json.dump(fused_pred, f, indent=2)

print(f"Fused predictions saved in {output_path}")


Fused predictions saved in C:\Medical_image_analysis\Polyp_domain_adaptation\preds\fused


In [21]:
import os
import json
import cv2

# Paths
fused_pred_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\fused"
image_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\combined_test\images"
output_vis_path = r"C:\Medical_image_analysis\Polyp_domain_adaptation\preds\visualizations"

os.makedirs(output_vis_path, exist_ok=True)

# Colors for classes (example for 5 classes)
colors = [(255,0,0), (0,255,0), (0,0,255), (255,255,0), (255,0,255)]

# List all images in the folder
all_images = {os.path.splitext(f)[0]: f for f in os.listdir(image_path)}

# List all fused JSONs
json_files = sorted(os.listdir(fused_pred_path))

for fname in json_files:
    # Load fused predictions
    with open(os.path.join(fused_pred_path, fname), 'r') as f:
        data = json.load(f)

    # Remove .json extension
    base_name = os.path.splitext(fname)[0]

    # Further remove .jpeg/.jpg/.png if present in JSON name
    for ext in ['.jpeg', '.jpg', '.png']:
        if base_name.lower().endswith(ext):
            base_name = base_name[:-len(ext)]

    # Find corresponding image
    img_file_name = all_images.get(base_name)
    if img_file_name is None:
        print(f"Image not found for {fname}, skipping.")
        continue

    img_file = os.path.join(image_path, img_file_name)
    img = cv2.imread(img_file)

    # Draw boxes
    for pred in data['preds']:
        xyxy = pred['xyxy']  # [x1, y1, x2, y2]
        cls = pred['cls']
        conf = pred['conf']
        color = colors[cls % len(colors)]
        cv2.rectangle(img, (int(xyxy[0]), int(xyxy[1])), (int(xyxy[2]), int(xyxy[3])), color, 2)
        text = f"Class:{cls} Conf:{conf:.2f}"
        cv2.putText(img, text, (int(xyxy[0]), int(xyxy[1])-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # Save visualization
    out_file = os.path.join(output_vis_path, os.path.splitext(fname)[0] + '.jpg')
    cv2.imwrite(out_file, img)

print(f"Visualization images saved in {output_vis_path}")


Visualization images saved in C:\Medical_image_analysis\Polyp_domain_adaptation\preds\visualizations
